In [4]:
%load_ext watermark
%watermark -a "Manuel Elias Orellana Lavayen" -d -v -iv

The watermark extension is already loaded. To reload it, use:
  %reload_ext watermark
Author: Manuel Elias Orellana Lavayen

Date: 2026-10-05

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.17.1

debugpy: 1.8.22
pandas : 3.0.5



## Importando Librerias

In [5]:
import pandas as pd
from pathlib import Path
ruta_base = Path.cwd().parent.parent
ruta_base

WindowsPath('C:/Users/ManuelOrellana/Desktop/Proyectos Aplicaciones/Aplicacion Prediccion Productos/Creacion de Modelos/Analisis y entrenamiento de modelos')

### Cargando Dataset

In [6]:
df_oil = pd.read_csv(filepath_or_buffer= ruta_base/"Datasets/Originales/oil.csv", parse_dates=["date"])
df_oil.head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


### Verificando Nulos por columnas

In [7]:
print("Nulos 'date': ", df_oil["date"].isnull().sum())
print("Nulos 'dcoilwtico': ", df_oil["dcoilwtico"].isnull().sum())

Nulos 'date':  0
Nulos 'dcoilwtico':  43


### Creando columnas de año y mes

In [8]:
df_oil["año"] = df_oil["date"].dt.year
df_oil["mes"] = df_oil["date"].dt.month
df_oil.head(2)

,date,dcoilwtico,año,mes
0,2013-01-01,NaN,2013,1
1,2013-01-02,93.14,2013,1


### Rellenando nulos con la mediana de ese año y de ese mes

In [9]:
df_oil["dcoilwtico"] = df_oil["dcoilwtico"].fillna(df_oil.groupby(["año", "mes"])["dcoilwtico"].transform("median"))

In [10]:
df_oil = df_oil.drop(columns= ["año", "mes"])

## Creación de filas para fines de semana en oil (Los fines de semana no estan contemplados en el dataset de oil)

### Dataframe de serie de tiempo con todos las fechas incluyendo fines de semana

In [11]:
serie_tiempo = pd.date_range(start=df_oil["date"].min() , end=df_oil["date"].max())
df_serie_tiempo = pd.DataFrame({"date": serie_tiempo})

### Uniendo Dataset de Oil con Serie de tiempo (Dejando los nulos en los fines de semana)

In [12]:
df_oil_2 = pd.merge(
    df_serie_tiempo,
    df_oil,
    on= "date",
    how= "left"
)

### Creacion de columnas con año, mes, numero de semana y dia de la semana

In [13]:
df_oil_2["año"] = df_oil_2["date"].dt.year
df_oil_2["mes"] = df_oil_2["date"].dt.month
df_oil_2["numero_semana"] = df_oil_2["date"].dt.isocalendar().week
df_oil_2["dia_semana"] = df_oil_2["date"].dt.dayofweek

df_oil_2.head(7)

,date,dcoilwtico,año,mes,numero_semana,dia_semana
0,2013-01-01,94.28,2013,1,1,1
1,2013-01-02,93.14,2013,1,1,2
2,2013-01-03,92.97,2013,1,1,3
3,2013-01-04,93.12,2013,1,1,4
4,2013-01-05,NaN,2013,1,1,5
5,2013-01-06,NaN,2013,1,1,6
6,2013-01-07,93.20,2013,1,2,0


### Rellenando nulos de los fines de semana en base a su valor del dia viernes, esto evita fuga de datos ya que usamos el ultimo valor conocido

In [14]:
df_oil_2["dcoilwtico"] = df_oil_2["dcoilwtico"].ffill()

df_oil_2.head(7)

,date,dcoilwtico,año,mes,numero_semana,dia_semana
0,2013-01-01,94.28,2013,1,1,1
1,2013-01-02,93.14,2013,1,1,2
2,2013-01-03,92.97,2013,1,1,3
3,2013-01-04,93.12,2013,1,1,4
4,2013-01-05,93.12,2013,1,1,5
5,2013-01-06,93.12,2013,1,1,6
6,2013-01-07,93.20,2013,1,2,0


### Guardamos el dataset

In [15]:
df_oil_2 = df_oil_2.drop(columns=["año", "mes", "mes", "numero_semana", "dia_semana"])
df_oil_2.to_csv(path_or_buf= ruta_base/ "Datasets/Modificados/oil_limpio.csv", index= False)